# Model A: TF-IDF + Linear Baseline
One-vs-rest Logistic Regression / LinearSVC trained on TF-IDF features with per-label threshold tuning.

## 1. Config

In [ ]:
# Load libraries and set paths for model training
# Define model type and basic training hyperparameters like regularization strength, max iterations and number of parallel jobs.
import numpy as np
import pandas as pd
import scipy.sparse as sp
import pickle, os, json
from pathlib import Path

DATA_DIR  = '../datasets/processed'
MODEL_DIR = '../data/models/model_a'
os.makedirs(MODEL_DIR, exist_ok=True)

CLASSIFIER     = 'sgd'
C_REG          = 1.0
MAX_ITER       = 100
N_JOBS         = -1

## 2. Load features & labels

In [ ]:
# Load preprocessed TF-IDF features and labels for train/val/test sets.
# Along with the label binarizer for decoding predictions of the model back to ICD codes.
X_train = sp.load_npz(f'{DATA_DIR}/X_train_tfidf.npz')
X_val   = sp.load_npz(f'{DATA_DIR}/X_val_tfidf.npz')
X_test  = sp.load_npz(f'{DATA_DIR}/X_test_tfidf.npz')
Y_train = np.load(f'{DATA_DIR}/Y_train.npy')
Y_val   = np.load(f'{DATA_DIR}/Y_val.npy')
Y_test  = np.load(f'{DATA_DIR}/Y_test.npy')

with open(f'{DATA_DIR}/mlb.pkl', 'rb') as f:
    mlb = pickle.load(f)
vocab = list(mlb.classes_)

print(f'X_train: {X_train.shape}  Y_train: {Y_train.shape}')
print(f'Labels: {len(vocab)}')

## 3. Train One-vs-Rest classifier

In [ ]:
# Train a One-vs-Rest classifier using the SGDClassifier with log loss (logistic regression) as the base estimator.
from sklearn.multiclass import OneVsRestClassifier
from sklearn.linear_model import LogisticRegression, SGDClassifier
from sklearn.svm import LinearSVC

if CLASSIFIER == 'sgd':
    base = SGDClassifier(
        loss='log_loss',
        penalty='l2',
        alpha=1.0 / (C_REG * X_train.shape[0]),
        max_iter=MAX_ITER,
        class_weight='balanced',
        tol=1e-3,
        random_state=42,
    )
    clf = OneVsRestClassifier(base, n_jobs=N_JOBS)
elif CLASSIFIER == 'lr':
    base = LogisticRegression(C=C_REG, max_iter=MAX_ITER, solver='liblinear',
                              class_weight='balanced')
    clf = OneVsRestClassifier(base, n_jobs=N_JOBS)
else:  # svm
    base = LinearSVC(C=C_REG, max_iter=MAX_ITER, class_weight='balanced')
    clf = OneVsRestClassifier(base, n_jobs=N_JOBS)

print(f'Training OvR on {X_train.shape[0]} samples × {len(vocab)} labels...')
clf.fit(X_train, Y_train)
print('Done.')

with open(f'{MODEL_DIR}/clf_{CLASSIFIER}.pkl', 'wb') as f:
    pickle.dump(clf, f)
print('Model saved.')

## 4. Probability estimates

In [ ]:
# LinearSVC: does not support predict_proba, so we will use decision_function outputs and apply sigmoid to get probabilities.
print('Computing probabilities on val set...')
if hasattr(clf, 'predict_proba'):
    P_val  = clf.predict_proba(X_val)
    P_test = clf.predict_proba(X_test)
else:
    
    from scipy.special import expit
    P_val  = expit(clf.decision_function(X_val))
    P_test = expit(clf.decision_function(X_test))
print(f'P_val shape: {P_val.shape}')

## 5. Threshold tuning on validation set

In [ ]:
# It is difficult to choose a single global threshold for all labels. 
# Using a range of thresholds, we can find the one that maximizes micro-F1 on the validation set.
from sklearn.metrics import f1_score

def tune_global_threshold(P, Y, thresholds=np.arange(0.05, 0.55, 0.025)):
    best_t, best_f1 = 0.5, 0.0
    for t in thresholds:
        preds = (P >= t).astype(int)
        f1 = f1_score(Y, preds, average='micro', zero_division=0)
        if f1 > best_f1:
            best_f1, best_t = f1, t
    return best_t, best_f1

best_t, best_f1_val = tune_global_threshold(P_val, Y_val)
print(f'Best global threshold: {best_t:.3f}  →  micro-F1 on val: {best_f1_val:.4f}')

## 6. Evaluation

In [ ]:
# With the best threshold, we can evaluate the model on both val and test sets using multiple metrics like:
# micro/macro F1, precision, recall, AUPRC, and AUROC.
from sklearn.metrics import (
    f1_score, precision_score, recall_score,
    average_precision_score, roc_auc_score
)

def evaluate(P, Y, threshold, split_name='test'):
    preds = (P >= threshold).astype(int)
    results = {
        'split'       : split_name,
        'threshold'   : threshold,
        'micro_f1'    : f1_score(Y, preds, average='micro',   zero_division=0),
        'macro_f1'    : f1_score(Y, preds, average='macro',   zero_division=0),
        'micro_prec'  : precision_score(Y, preds, average='micro', zero_division=0),
        'micro_rec'   : recall_score(Y, preds, average='micro',    zero_division=0),
    }
    # AUPRC / AUROC — only for labels with positives in this split
    mask = Y.sum(0) > 0
    results['macro_auprc'] = average_precision_score(Y[:, mask], P[:, mask], average='macro')
    results['micro_auroc'] = roc_auc_score(Y[:, mask], P[:, mask], average='micro')
    for k, v in results.items():
        if isinstance(v, float):
            print(f'  {k:20s}: {v:.4f}')
        else:
            print(f'  {k:20s}: {v}')
    return results

print('=== Validation ===')
val_results  = evaluate(P_val,  Y_val,  best_t, 'val')
print('\n=== Test ===')
test_results = evaluate(P_test, Y_test, best_t, 'test')

with open(f'{MODEL_DIR}/results.json', 'w') as f:
    json.dump({'val': val_results, 'test': test_results}, f, indent=2)
print('\nResults saved.')

## 7. Head vs. tail label analysis

In [ ]:
import matplotlib.pyplot as plt

# Per-label F1 on test set vs training frequency plot
preds_test = (P_test >= best_t).astype(int)
per_label_f1   = f1_score(Y_test, preds_test, average=None, zero_division=0)
per_label_freq = Y_train.sum(0)   # training frequency

label_df = pd.DataFrame({
    'icd_code'   : vocab,
    'train_freq' : per_label_freq,
    'test_f1'    : per_label_f1
}).sort_values('train_freq', ascending=False)


thresholds_freq = [500, 100, 0]
labels_bucket   = ['head (≥500)', 'torso (100-499)', 'tail (<100)']

for bucket, (lo, hi) in zip(labels_bucket, [(500, 1e9), (100, 499), (0, 99)]):
    subset = label_df[(label_df['train_freq'] >= lo) & (label_df['train_freq'] <= hi)]
    print(f'{bucket:25s}  n_codes={len(subset):5d}  avg_F1={subset["test_f1"].mean():.4f}')

# Scatter: frequency vs F1
plt.figure(figsize=(7, 4))
plt.scatter(label_df['train_freq'].clip(upper=2000), label_df['test_f1'], alpha=0.3, s=8)
plt.xscale('log'); plt.xlabel('Train freq (log)'); plt.ylabel('Test F1')
plt.title('Per-label F1 vs training frequency (Model A)')
plt.tight_layout(); plt.savefig(f'{MODEL_DIR}/head_tail_f1.png', dpi=120); plt.show()

## 8. Top-K evaluation (Top-50, Top-100, Top-500)

In [ ]:
# Performance on top-K most frequent labels.
for K in [50, 100, 500]:
    top_k_idx = label_df.nlargest(K, 'train_freq').index
    Y_k = Y_test[:, top_k_idx]
    P_k = P_test[:, top_k_idx]
    preds_k = (P_k >= best_t).astype(int)
    mf1 = f1_score(Y_k, preds_k, average='micro', zero_division=0)
    print(f'Top-{K:4d}  micro-F1 = {mf1:.4f}')

## 9. Confusion Matrix (Per-Label)

In [ ]:
#Plots confusion matrices for top 10 labels
import numpy as np
import matplotlib.pyplot as plt
import pickle, json, os
import scipy.sparse as sp
from sklearn.metrics import multilabel_confusion_matrix

DATA_DIR  = '../datasets/processed'
MODEL_DIR = '../data/models/model_a'
TOP_N = 10

with open(f'{DATA_DIR}/mlb.pkl', 'rb') as f:
    mlb = pickle.load(f)
vocab = list(mlb.classes_)

Y_test = np.load(f'{DATA_DIR}/Y_test.npy')


P_test_path = f'{MODEL_DIR}/P_test.npy'
P_test = np.load(P_test_path)

if P_test.shape[0] != Y_test.shape[0]:
    print('Length mismatch')

    X_test = sp.load_npz(f'{DATA_DIR}/X_test_tfidf.npz')

    clf = None
    for name in ['clf_sgd.pkl', 'clf_lr.pkl', 'clf_svm.pkl']:
        p = f'{MODEL_DIR}/{name}'
        if os.path.exists(p):
            with open(p, 'rb') as f:
                clf = pickle.load(f)
            print(f'Loaded classifier: {p}')
            break

    if clf is None:
        raise FileNotFoundError(
            'No saved classifier'
        )

    if hasattr(clf, 'predict_proba'):
        P_test = clf.predict_proba(X_test)
    else:
        from scipy.special import expit
        P_test = expit(clf.decision_function(X_test))

    np.save(P_test_path, P_test)
    print(f'Saved updated P_test to {P_test_path} with shape {P_test.shape}.')
with open(f'{MODEL_DIR}/results.json') as f:
    threshold = json.load(f)['test']['threshold']
preds_test = (P_test >= threshold).astype(int)

mcm = multilabel_confusion_matrix(Y_test, preds_test)

freq = Y_test.sum(0)
top_idx = np.argsort(freq)[::-1][:TOP_N]

nrows, ncols = 2, TOP_N // 2
fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 4 * nrows))

for idx, ax in zip(top_idx, axes.flat):
    cm = mcm[idx]
    ax.imshow(cm, cmap='Blues')
    ax.set_title(vocab[idx], fontsize=10, fontweight='bold')
    ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
    ax.set_xticklabels(['Neg', 'Pos']); ax.set_yticklabels(['Neg', 'Pos'])
    ax.set_xlabel('Predicted'); ax.set_ylabel('Actual')
    for r in range(2):
        for c in range(2):
            color = 'white' if cm[r, c] > cm.max() / 2 else 'black'
            row_total = cm[r].sum()
            pct = cm[r, c] / row_total * 100 if row_total > 0 else 0
            ax.text(c, r, f'{cm[r, c]:,}\n({pct:.1f}%)', ha='center', va='center', color=color, fontsize=11)

plt.suptitle(f'Per-Label Confusion Matrices — Model A (TF-IDF + SGD), Top {TOP_N} Labels', fontsize=14)
plt.tight_layout()
plt.savefig(f'{MODEL_DIR}/confusion_matrix_top{TOP_N}.png', dpi=150)
plt.show()

agg = mcm.sum(axis=0)
print(f'\nAggregate Confusion Matrix (all {len(vocab)} labels):')
agg_total = agg.sum()
print(f'  TN = {agg[0,0]:>10,} ({agg[0,0]/agg_total*100:.1f}%)   FP = {agg[0,1]:>10,} ({agg[0,1]/agg_total*100:.1f}%)')
print(f'  FN = {agg[1,0]:>10,} ({agg[1,0]/agg_total*100:.1f}%)   TP = {agg[1,1]:>10,} ({agg[1,1]/agg_total*100:.1f}%)')

## 10. Full 50-Label Confusion Analysis
For each pair of labels (i, j): when label i is truly present in a sample, how often does the model **falsely** predict label j on that same sample? This reveals which ICD codes the model confuses with each other.

In [ ]:
# 50 label confusion heatmap
# To analyze common confusions between labels, we can create a heatmap of how often each true label is falsely predicted as another label.
# Considering top 50 most frequent labels the map is very difficult to read.
TOP_PAIRS = 20
n_labels = len(vocab)

fp_mask = (preds_test == 1) & (Y_test == 0)
tp_mask = (Y_test == 1)

label_confusion = tp_mask.astype(int).T @ fp_mask.astype(int)

fig, ax = plt.subplots(figsize=(18, 15))
row_sums = tp_mask.sum(axis=0).reshape(-1, 1)
row_sums = np.where(row_sums == 0, 1, row_sums)
label_confusion_pct = label_confusion / row_sums * 100
im = ax.imshow(label_confusion_pct, cmap='Reds', aspect='auto', interpolation='nearest')
ax.set_xticks(range(n_labels)); ax.set_yticks(range(n_labels))
ax.set_xticklabels(vocab, rotation=90, fontsize=7)
ax.set_yticklabels(vocab, fontsize=7)
ax.set_xlabel('Falsely Predicted Label (FP)', fontsize=12)
ax.set_ylabel('Truly Present Label (TP)', fontsize=12)
ax.set_title(f'Model A — {n_labels}-Label Confusion Heatmap (%)', fontsize=14)
plt.colorbar(im, ax=ax, shrink=0.8, label='% of true-label samples')
plt.tight_layout()
plt.savefig(f'{MODEL_DIR}/label_confusion_{n_labels}x{n_labels}.png', dpi=150)
plt.show()

np.fill_diagonal(label_confusion, 0)
flat_idx = np.argsort(label_confusion.ravel())[::-1]

print(f'\nTop {TOP_PAIRS} most confused label pairs (Model A):')
print(f'  {"True Label":>12s}  →  {"Falsely Predicted":>18s}  {"Count":>7s}')
print('  ' + '-' * 50)
shown = 0
for fi in flat_idx:
    if shown >= TOP_PAIRS:
        break
    i, j = divmod(fi, n_labels)
    count = label_confusion[i, j]
    if count == 0:
        break
    tp_total = tp_mask[:, i].sum()
    pct = count / tp_total * 100 if tp_total > 0 else 0
    print(f'  {vocab[i]:>12s}  →  {vocab[j]:>18s}  {count:>7,}  ({pct:.1f}% of {vocab[i]} samples)')
    shown += 1